# HoVer v6 Hop2 Refill — 학습 및 평가 전용

이 노트북은 HoVer v6 판정 모델을 재현하고 평가하는 학습용 노트북입니다. FastAPI 서버, ngrok 터널, 앱 인증 토큰 코드는 포함하지 않습니다.

## 실행 순서

1. 셀 2–32: 데이터·문장 캐시·v5 근거 생성 결과 및 기존 판정기 준비. 데이터와 캐시가 이미 Google Drive에 있으면 안내에 따라 해당 준비 단계를 건너뛸 수 있습니다.
2. 셀 33: Hop 1과 겹치는 Hop 2 근거를 제외했을 때의 validation 비교.
3. 셀 34: Hop 2의 중복 후보를 빼고 다음 순위 후보로 채우는 재검색 및 validation 비교.
4. 셀 35: v6 판정기 학습, validation 평가, Google Drive에 체크포인트 저장.

23번과 26번은 v5 전체 근거를 처음부터 다시 생성하는 셀이므로, 저장된 근거 캐시를 사용할 때는 실행하지 마세요.

학습 체크포인트 저장 경로: `MyDrive/hover_final_verifier_v6_hop2_refill/`. 실행이 끝나면 폴더 안의 모델 파일을 별도로 배포해야 다른 사용자가 재학습 없이 모델을 불러올 수 있습니다. 배포 파일 안내는 `colab/README_V6_RELEASE.md`를 참고하세요.


## 1. Google Drive 연결 및 런타임 준비

In [ ]:
# 2번: Drive를 연결하고 HoVer 데이터 및 위키 DB를 준비합니다.
from google.colab import drive
from pathlib import Path
import os
import subprocess

# 런타임 재연결 후 /content/drive가 이미 차 있어도 기존 파일을 지우지 않고
# 별도의 빈 마운트 지점을 사용합니다.
DRIVE_MOUNT = Path("/content/hover_drive_mount")
if not os.path.ismount(str(DRIVE_MOUNT)):
    DRIVE_MOUNT.mkdir(parents=True, exist_ok=True)
    if any(DRIVE_MOUNT.iterdir()):
        raise RuntimeError(f"Drive 마운트 지점이 비어 있지 않습니다: {DRIVE_MOUNT}")
    drive.mount(str(DRIVE_MOUNT))
else:
    print("Google Drive 이미 연결됨:", DRIVE_MOUNT)
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive"

BASE = Path("/content/hover")
if not BASE.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/hover-nlp/hover.git", str(BASE)],
        check=True
    )
if not (BASE / "data/wiki_wo_links.db").exists():
    subprocess.run(["bash", "download_data.sh"], cwd=BASE, check=True)

required = [
    BASE / "data/hover/hover_train_release_v1.1.json",
    BASE / "data/hover/tfidf_retrieved/train_tfidf_doc_retrieval_results.json",
    BASE / "data/wiki_wo_links.db",
]
for path in required:
    print("있음" if path.exists() else "없음", path)

In [ ]:
# 3번: 필요한 NLP 패키지와 spaCy 영어 모델을 준비합니다.
%pip -q install "huggingface_hub>=1.32,<2" sentence-transformers spacy tqdm scikit-learn

import spacy
import sys
if not spacy.util.is_package("en_core_web_sm"):
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
nlp = spacy.load("en_core_web_sm")
print("spaCy 영어 모델 준비 완료")

## 2. CoreNLP 설치 및 문장 분리 서버 실행

In [ ]:
# 5번: CoreNLP 문장 분리 도구를 다운로드하고 압축을 풉니다.
from pathlib import Path
import zipfile
from huggingface_hub import hf_hub_download

CORENLP_DIR = Path("/content/corenlp-unpacked/stanford-corenlp-4.6.0")
if not CORENLP_DIR.exists():
    zip_path = Path(hf_hub_download(
        repo_id="stanfordnlp/CoreNLP",
        filename="stanford-corenlp-latest.zip",
        local_dir="/content/corenlp-download",
    ))
    if not zipfile.is_zipfile(zip_path):
        raise RuntimeError(f"CoreNLP ZIP 파일이 아닙니다: {zip_path}")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall("/content/corenlp-unpacked")

print("CoreNLP 폴더:", CORENLP_DIR.exists())
print("JAR 파일 수:", len(list(CORENLP_DIR.glob("*.jar"))))

In [ ]:
# 6번: CoreNLP 문장 분리 서버를 실행하고 연결을 확인합니다.
import json
import subprocess
import time
import requests

def corenlp_is_ready():
    try:
        r = requests.post(
            "http://localhost:9001",
            params={"properties": json.dumps({"annotators": "tokenize,ssplit", "outputFormat": "json"})},
            data="One sentence. Another sentence.",
            timeout=5,
        )
        return r.ok
    except requests.RequestException:
        return False

if not corenlp_is_ready():
    corenlp_log = open("/content/corenlp_server_9001.log", "w")
    corenlp_process = subprocess.Popen(
        ["java", "-mx4g", "-cp", "*",
         "edu.stanford.nlp.pipeline.StanfordCoreNLPServer",
         "-annotators", "tokenize,ssplit", "-port", "9001",
         "-threads", "4", "-timeout", "60000"],
        cwd=str(CORENLP_DIR), stdout=corenlp_log, stderr=corenlp_log,
    )
    for _ in range(30):
        if corenlp_is_ready():
            break
        time.sleep(2)

if not corenlp_is_ready():
    raise RuntimeError("CoreNLP 서버가 9001 포트에서 응답하지 않습니다.")
print("CoreNLP 서버 연결 성공")

In [ ]:
# 7번: HoVer TF-IDF 문서 검색 결과 파일을 준비합니다.
from pathlib import Path
from urllib.request import urlopen
import json
import shutil

retrieval_path = BASE / "data/hover/tfidf_retrieved/train_tfidf_doc_retrieval_results.json"
retrieval_path.parent.mkdir(parents=True, exist_ok=True)

url = "https://nlp.cs.unc.edu/data/hover/train_tfidf_doc_retrieval_results.json"
temp_path = retrieval_path.with_suffix(".json.part")

print("다운로드 중...")
with urlopen(url, timeout=120) as response, open(temp_path, "wb") as out:
    shutil.copyfileobj(response, out)

# 다운로드된 파일이 정상 JSON인지 확인한 뒤 최종 경로로 옮깁니다.
with open(temp_path, encoding="utf-8") as f:
    retrieval_check = json.load(f)

if not isinstance(retrieval_check, list):
    raise ValueError("파일은 다운로드됐지만 예상한 JSON 리스트 형식이 아닙니다.")

temp_path.replace(retrieval_path)

print("완료:", retrieval_path)
print("검색 결과 행 수:", len(retrieval_check))
print(f"파일 크기: {retrieval_path.stat().st_size / 1024**2:.1f} MB")

## 3. HoVer 2-hop 데이터, split, TF-IDF Top-50 연결

In [ ]:
# 9번: HoVer 2-hop 데이터를 나누고 TF-IDF Top-50 문서 목록을 연결합니다.
import json
import sqlite3
import unicodedata
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

train_path = BASE / "data/hover/hover_train_release_v1.1.json"
retrieval_path = BASE / "data/hover/tfidf_retrieved/train_tfidf_doc_retrieval_results.json"
db_path = BASE / "data/wiki_wo_links.db"

# TF-IDF 파일이 예상 위치에 없으면 현재 런타임의 데이터 폴더에서 검색
if not retrieval_path.exists():
    filename = "train_tfidf_doc_retrieval_results.json"
    search_roots = [
        BASE,
        Path("/content/hover"),
        Path("/content/drive/MyDrive/hover"),
        Path("/content/hover_drive_mount/MyDrive/hover"),
    ]

    matches = []
    checked = set()
    for root in search_roots:
        if root.exists() and str(root) not in checked:
            checked.add(str(root))
            matches.extend(root.rglob(filename))

    # 중복 경로 제거
    matches = list(dict.fromkeys(matches))

    if matches:
        retrieval_path = matches[0]
    else:
        raise FileNotFoundError(
            f"{filename} 파일을 찾지 못했습니다. "
            f"처음 확인한 경로: {BASE / 'data/hover/tfidf_retrieved' / filename}"
        )

print("Train 파일:", train_path, "| 존재:", train_path.exists())
print("검색 결과 파일:", retrieval_path, "| 존재:", retrieval_path.exists())
print("위키 DB:", db_path, "| 존재:", db_path.exists())

with open(train_path, encoding="utf-8") as f:
    train_raw = json.load(f)

with open(retrieval_path, encoding="utf-8") as f:
    retrieval_raw = json.load(f)

conn = sqlite3.connect(str(db_path), check_same_thread=False)
retrieval_by_claim = {row["claim"]: row for row in retrieval_raw}

two_hop_data = [row for row in train_raw if row["num_hops"] == 2]
train_data, val_data = train_test_split(
    two_hop_data,
    test_size=0.2,
    random_state=42,
    stratify=[row["label"] for row in two_hop_data],
)

DOC_TOP_K = 50
titles_by_claim_top50 = {}
all_titles_top50 = set()

for row in train_data + val_data:
    titles = retrieval_by_claim[row["claim"]]["doc_retrieval_results"][0][0][:DOC_TOP_K]
    titles_by_claim_top50[row["claim"]] = titles
    all_titles_top50.update(titles)

print("2-hop 전체:", len(two_hop_data))
print("학습:", len(train_data), Counter(r["label"] for r in train_data))
print("검증:", len(val_data), Counter(r["label"] for r in val_data))
print("Top-50 고유 문서:", len(all_titles_top50))

## 4. CoreNLP 문장 캐시 만들기 또는 Drive 캐시 불러오기

In [ ]:
# 11번: CoreNLP로 문장을 나누고 Drive의 문장 캐시를 불러오거나 저장합니다.
import html
import re
import pickle
import unicodedata
from functools import lru_cache
from tqdm.auto import tqdm

SENTENCE_CACHE_PATH = DRIVE_ROOT / "hover_v5_top50_sentence_cache.pkl"

def normalize_title(title):
    title = unicodedata.normalize("NFKD", str(title))
    title = "".join(ch for ch in title if not unicodedata.combining(ch))
    return " ".join(title.casefold().split())

def clean_sentence(text):
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", html.unescape(text)).strip()

@lru_cache(maxsize=5000)
def get_sentences_corenlp(title):
    title_forms = list(dict.fromkeys([
        unicodedata.normalize("NFD", title),
        unicodedata.normalize("NFC", title),
    ]))
    result = None
    for title_form in title_forms:
        result = conn.execute(
            "SELECT text FROM documents WHERE id = ?", (title_form,)
        ).fetchone()
        if result is not None:
            break
    if result is None:
        return ()

    raw_text = result[0]
    response = requests.post(
        "http://localhost:9001",
        params={"properties": json.dumps({"annotators": "tokenize,ssplit", "outputFormat": "json"})},
        data=raw_text.encode("utf-8"),
        headers={"Content-Type": "text/plain; charset=utf-8"},
        timeout=60,
    )
    response.raise_for_status()
    output = []
    for sentence in response.json()["sentences"]:
        tokens = sentence["tokens"]
        start = tokens[0]["characterOffsetBegin"]
        end = tokens[-1]["characterOffsetEnd"]
        output.append(raw_text[start:end].strip())
    return tuple(output)

if SENTENCE_CACHE_PATH.exists():
    with open(SENTENCE_CACHE_PATH, "rb") as f:
        candidate_sentences = pickle.load(f)
    print("Drive 문장 캐시 불러옴:", len(candidate_sentences))
else:
    candidate_sentences = {}

missing_cache_titles = sorted(all_titles_top50 - set(candidate_sentences))
cache_errors = []
for title in tqdm(missing_cache_titles, desc="Top-50 문서 문장 분리"):
    try:
        candidate_sentences[title] = get_sentences_corenlp(title)
    except Exception as exc:
        cache_errors.append((title, repr(exc)))
print("이번에 처리한 문서:", len(missing_cache_titles), "오류:", len(cache_errors))
if cache_errors:
    print("오류 예시:", cache_errors[:3])
with open(SENTENCE_CACHE_PATH, "wb") as f:
    pickle.dump(candidate_sentences, f, protocol=pickle.HIGHEST_PROTOCOL)
print("Drive 문장 캐시 저장:", SENTENCE_CACHE_PATH)

def get_title_sentences(title):
    if title not in candidate_sentences:
        candidate_sentences[title] = get_sentences_corenlp(title)
    return candidate_sentences[title]

## 5. 사전학습 Cross-Encoder 재순위기와 spaCy 개체 추출

In [ ]:
# 13번: Cross-Encoder 문장 재순위 모델과 점수 함수를 준비합니다.
import numpy as np
import torch
from sentence_transformers import CrossEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
base_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    num_labels=1,
    max_length=256,
    device=device,
)
print("문장 재순위기 로드:", device)

def score_sentence_pairs_fast(pairs, chunk_size=10000, batch_size=128):
    if not pairs:
        return np.array([])
    scores = []
    for start in range(0, len(pairs), chunk_size):
        batch = pairs[start:start + chunk_size]
        batch_scores = base_encoder.predict(
            batch, batch_size=batch_size, show_progress_bar=False
        )
        scores.extend(np.asarray(batch_scores).reshape(-1))
    return np.asarray(scores)

allowed_entity_labels = {"PERSON", "ORG", "GPE", "LOC", "FAC"}

In [ ]:
# 14번: 런타임 GPU가 문장 재순위 모델에 연결됐는지 확인합니다.
import torch, shutil, subprocess, os
print("torch:", torch.__version__, "CUDA build:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("nvidia-smi path:", shutil.which("nvidia-smi"))
print("/dev/nvidia devices:", [p for p in os.listdir("/dev") if p.startswith("nvidia")])
if shutil.which("nvidia-smi"):
    probe = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(probe.stdout[:2000])

## 6. 2-hop 제목 매칭 및 후보 생성 함수 (v5)

In [ ]:
# 16번: Hop 1·Hop 2 근거 후보 생성 함수를 정의합니다.
import re

def normalize_for_title_match(text):
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9가-힣\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def remove_parenthetical(text):
    return re.sub(r"\s*\([^)]*\)", "", str(text)).strip()

def lookup_entity_titles(entity_name, max_results=20):
    entity_name = str(entity_name).strip()
    if not entity_name:
        return []
    candidates = []
    exact_rows = conn.execute(
        "SELECT id FROM documents WHERE id = ? LIMIT 20", (entity_name,)
    ).fetchall()
    candidates.extend(row[0] for row in exact_rows)

    escaped = entity_name.replace("*", "[*]").replace("?", "[?]")
    prefix_rows = conn.execute(
        "SELECT id FROM documents WHERE id GLOB ? LIMIT ?",
        (escaped + "*", max_results * 3),
    ).fetchall()
    candidates.extend(row[0] for row in prefix_rows)

    unique = list(dict.fromkeys(candidates))
    entity_norm = normalize_for_title_match(entity_name)
    entity_base = normalize_for_title_match(remove_parenthetical(entity_name))
    ranked = []
    for title in unique:
        title_norm = normalize_for_title_match(title)
        title_base = normalize_for_title_match(remove_parenthetical(title))
        if title_norm == entity_norm:
            priority = 0
        elif title_base == entity_base:
            priority = 1
        elif title_norm.startswith(entity_norm):
            priority = 2
        elif entity_norm in title_norm:
            priority = 3
        else:
            priority = 4
        ranked.append((priority, len(title), title))
    ranked.sort(key=lambda x: (x[0], x[1]))
    return [title for _, _, title in ranked[:max_results]]

def build_explicit_two_hop_v5(row, hop1_top_k=5, hop2_top_k=5, max_hop2_titles=100):
    claim = row["claim"]
    if claim not in titles_by_claim_top50:
        raise KeyError("이 claim은 현재 TF-IDF 검색 목록에 없습니다.")

    # Hop 1: Top-50 문서의 전체 문장을 Cross-Encoder로 재순위
    hop1_candidates = []
    for title in titles_by_claim_top50[claim][:50]:
        for sentence_id, sentence in enumerate(get_title_sentences(title)):
            cleaned = clean_sentence(sentence)
            if cleaned:
                hop1_candidates.append({
                    "title": title,
                    "sentence_id": sentence_id,
                    "sentence": cleaned,
                })
    if not hop1_candidates:
        return {"claim": claim, "hop1": [], "entities": [], "hop2": []}

    hop1_scores = score_sentence_pairs_fast([
        (claim, item["sentence"]) for item in hop1_candidates
    ])
    for item, score in zip(hop1_candidates, hop1_scores):
        item["score"] = float(score)
    hop1_ranked = sorted(hop1_candidates, key=lambda x: x["score"], reverse=True)[:hop1_top_k]

    # Claim과 Hop 1 문장에서 연결 개체 추출
    entities = []
    for text in [claim] + [item["sentence"] for item in hop1_ranked]:
        for entity in nlp(text).ents:
            if entity.label_ in allowed_entity_labels:
                name = entity.text.strip()
                if len(name) >= 2 and name not in entities:
                    entities.append(name)

    # Hop 2 문서: 연결 개체의 제목 후보 + Hop 1 문서 + Top-50 fallback
    hop2_titles, seen_titles = [], set()
    def add_title(title):
        key = normalize_title(title)
        if key and key not in seen_titles:
            seen_titles.add(key)
            hop2_titles.append(title)

    for item in hop1_ranked:
        add_title(item["title"])
    for entity_name in entities:
        for title in lookup_entity_titles(entity_name, max_results=20):
            add_title(title)
    if len(hop2_titles) < 20:
        for title in titles_by_claim_top50[claim][:50]:
            add_title(title)
    hop2_titles = hop2_titles[:max_hop2_titles]

    hop2_candidates = []
    seen_facts = set()
    for title in hop2_titles:
        for sentence_id, sentence in enumerate(get_title_sentences(title)):
            fact = (normalize_title(title), sentence_id)
            cleaned = clean_sentence(sentence)
            if fact not in seen_facts and cleaned:
                seen_facts.add(fact)
                hop2_candidates.append({
                    "title": title,
                    "sentence_id": sentence_id,
                    "sentence": cleaned,
                })

    if not hop2_candidates:
        return {"claim": claim, "hop1": hop1_ranked, "entities": entities, "hop2": []}

    hop1_text = "\n".join(item["sentence"] for item in hop1_ranked)
    hop2_query = f"Claim: {claim}\nHop 1 evidence:\n{hop1_text}"
    hop2_scores = score_sentence_pairs_fast([
        (hop2_query, item["sentence"]) for item in hop2_candidates
    ])
    for item, score in zip(hop2_candidates, hop2_scores):
        item["score"] = float(score)
    hop2_ranked = sorted(hop2_candidates, key=lambda x: x["score"], reverse=True)[:hop2_top_k]

    return {"claim": claim, "hop1": hop1_ranked, "entities": entities, "hop2": hop2_ranked}

## 7. Train/validation 2-hop 근거 생성

*   항목 추가
*   항목 추가



In [ ]:
from pathlib import Path
from urllib.request import urlopen
import shutil
import sqlite3

new_db_path = Path("/content/wiki_wo_links_repaired.db")
url = "https://nlp.cs.unc.edu/data/hover/wiki_wo_links.db"

free_gb = shutil.disk_usage("/content").free / 1024**3
print(f"/content 남은 공간: {free_gb:.2f} GB")

if free_gb < 2.5:
    raise RuntimeError("다운로드 공간이 부족합니다. 기존 DB는 삭제하지 마세요.")

if new_db_path.exists():
    raise FileExistsError(f"대상 파일이 이미 있습니다: {new_db_path}")

print("새 DB 다운로드 중...")
with urlopen(url, timeout=180) as response, open(new_db_path, "wb") as out:
    shutil.copyfileobj(response, out)

print(f"다운로드 완료: {new_db_path.stat().st_size / 1024**3:.2f} GB")

# 새 파일이 정상인지 확인
check_conn = sqlite3.connect(
    f"{new_db_path.resolve().as_uri()}?mode=ro",
    uri=True
)
print("quick_check:", check_conn.execute("PRAGMA quick_check").fetchone())
print("테이블:", check_conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
).fetchall())
print("문서 샘플:", check_conn.execute(
    "SELECT id FROM documents LIMIT 1"
).fetchone())
check_conn.close()

In [ ]:
db_path = str(new_db_path)

In [ ]:
from pathlib import Path
import sqlite3

print("DB 경로:", db_path)
print("파일 존재:", Path(db_path).exists())
print("파일 크기(GB):", round(Path(db_path).stat().st_size / 1024**3, 2))

# 읽기 전용으로 새 연결을 열어 DB 상태 확인
uri = Path(db_path).resolve().as_uri() + "?mode=ro"
check_conn = sqlite3.connect(uri, uri=True)

print("quick_check:", check_conn.execute("PRAGMA quick_check").fetchone())

# 오류가 난 문서들을 직접 읽어보기
for title in ["Fas'h mosque", "Al-Masjid an-Nabawi"]:
    try:
        result = check_conn.execute(
            "SELECT length(text) FROM documents WHERE id = ?",
            (title,)
        ).fetchone()
        print(title, "→", result)
    except Exception as e:
        print(title, "→ 오류:", repr(e))

check_conn.close()

In [ ]:
import sqlite3

db_path = "/content/wiki_wo_links_repaired.db"
conn = sqlite3.connect(db_path, check_same_thread=False)
cursor = conn.cursor()

print(conn.execute("PRAGMA quick_check").fetchone())
print(conn.execute(
    "SELECT length(text) FROM documents WHERE id = ?",
    ("Fas'h mosque",)
).fetchone())

In [ ]:
test_row = next(
    row for row in train_data
    if row["claim"].startswith("Fash Mosque is 4.5 km")
)

test_result = build_explicit_two_hop_v5(test_row)

print("Claim:", test_result["claim"])
print("Hop 1:", test_result.get("hop1"))
print("Hop 2:", test_result.get("hop2"))

In [ ]:
from tqdm.auto import tqdm
import pickle

def build_dataset(data, description):
    results, errors = [], []

    for row in tqdm(data, desc=description):
        try:
            result = build_explicit_two_hop_v5(row)
            result["uid"] = row["uid"]
            result["label"] = row["label"]
            result["supporting_facts"] = row["supporting_facts"]
            results.append(result)
        except Exception as exc:
            errors.append({
                "uid": row.get("uid"),
                "claim": row.get("claim"),
                "error": repr(exc),
            })

    return results, errors


train_two_hop_v5, train_errors_v5 = build_dataset(
    train_data, "v5 학습 근거 생성"
)
val_two_hop_v5, val_errors_v5 = build_dataset(
    val_data, "v5 검증 근거 생성"
)

print("학습 결과/오류:", len(train_two_hop_v5), len(train_errors_v5))
print("검증 결과/오류:", len(val_two_hop_v5), len(val_errors_v5))

if train_errors_v5 or val_errors_v5:
    print("오류 예시:", (train_errors_v5 + val_errors_v5)[:3])
    raise RuntimeError("오류가 남아 있어요. 이 상태로 학습을 진행하지 마세요.")

with open(SENTENCE_CACHE_PATH, "wb") as f:
    pickle.dump(candidate_sentences, f, protocol=pickle.HIGHEST_PROTOCOL)

print("근거 생성 완료 및 문장 캐시 저장")

In [ ]:
import pickle

needed = [
    "train_two_hop_v5", "val_two_hop_v5",
    "train_errors_v5", "val_errors_v5", "SENTENCE_CACHE_PATH"
]
missing = [name for name in needed if name not in globals()]

if missing:
    print("현재 런타임에 없는 변수:", missing)
    print("화면에 예전 실행 결과가 보여도 런타임이 초기화됐다면 변수는 복구되지 않습니다.")
elif train_errors_v5 or val_errors_v5:
    print("오류가 남아 있어 결과를 저장하지 않았습니다.")
else:
    result_path = SENTENCE_CACHE_PATH.parent / "hover_v5_two_hop_results.pkl"

    if result_path.exists():
        print("결과 캐시가 이미 있습니다:", result_path)
    else:
        with open(result_path, "wb") as f:
            pickle.dump(
                {"train": train_two_hop_v5, "val": val_two_hop_v5},
                f,
                protocol=pickle.HIGHEST_PROTOCOL,
            )
        print("생성 결과를 Drive에 저장했습니다:", result_path)
        print("학습/검증 샘플:", len(train_two_hop_v5), len(val_two_hop_v5))

In [ ]:
# 25번: Drive에 저장된 v5 근거 캐시를 불러옵니다.
#런타임 때 실행 시 위위에 코드 안 해도 됨


import pickle

RESULT_CACHE_PATH = "/content/hover_drive_mount/MyDrive/hover_v5_two_hop_results.pkl"

with open(RESULT_CACHE_PATH, "rb") as f:
    cached_results = pickle.load(f)

train_two_hop_v5 = cached_results["train"]
val_two_hop_v5 = cached_results["val"]

print("학습/검증 샘플:", len(train_two_hop_v5), len(val_two_hop_v5))

In [ ]:
# v5 2-hop 근거 생성: 체크포인트 재개 + Drive 일괄 저장
import json, time
from pathlib import Path
from tqdm.auto import tqdm

# 런타임 재시작 후 이 셀만 다시 실행해도 필요한 spaCy 변수가 준비되도록 합니다.
if "nlp" not in globals():
    import spacy
    nlp = spacy.load("en_core_web_sm")
if "allowed_entity_labels" not in globals():
    allowed_entity_labels = {"PERSON", "ORG", "GPE", "LOC", "FAC"}

CHECKPOINT_DIR = DRIVE_ROOT / "hover_v5_hop_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
HOP2_CACHE_PATH = DRIVE_ROOT / "hover_v5_hop2_sentence_cache.jsonl"

# 기존 Hop-2 문장 캐시 복원
if HOP2_CACHE_PATH.exists():
    restored = 0
    with open(HOP2_CACHE_PATH, "r", encoding="utf-8") as f:
        for line in f:
            try:
                item = json.loads(line)
                title = item.get("title")
                if title and title not in candidate_sentences:
                    candidate_sentences[title] = tuple(item.get("sentences", []))
                    restored += 1
            except (json.JSONDecodeError, KeyError, TypeError):
                continue
    print("Drive Hop-2 문장 캐시 복원:", restored)

# Drive FUSE에 한 줄마다 flush하지 않고, 새 캐시 100건마다 묶어서 기록합니다.
_hop2_cache_buffer = []
_hop2_cached_now = set()
def _flush_hop2_cache_buffer():
    global _hop2_cache_buffer
    if not _hop2_cache_buffer:
        return
    with open(HOP2_CACHE_PATH, "a", encoding="utf-8") as f:
        f.writelines(json.dumps(x, ensure_ascii=False) + "\n" for x in _hop2_cache_buffer)
    _hop2_cache_buffer = []

def get_title_sentences_resumable(title):
    if title not in candidate_sentences:
        sentences = tuple(get_sentences_corenlp(title))
        candidate_sentences[title] = sentences
        if title not in _hop2_cached_now:
            _hop2_cache_buffer.append({"title": title, "sentences": sentences})
            _hop2_cached_now.add(title)
            if len(_hop2_cache_buffer) >= 100:
                _flush_hop2_cache_buffer()
    return candidate_sentences[title]
get_title_sentences = get_title_sentences_resumable

print("CUDA available:", torch.cuda.is_available(), "| reranker:", device)
if device != "cuda" or not torch.cuda.is_available():
    raise RuntimeError("GPU가 재순위 모델에 연결되지 않았습니다.")

def build_dataset_resumable(data, split_name, max_new=None, checkpoint_every=50):
    checkpoint_path = CHECKPOINT_DIR / f"{split_name}_v5.jsonl"
    error_path = CHECKPOINT_DIR / f"{split_name}_errors_v5.jsonl"
    completed = {}
    if checkpoint_path.exists():
        with open(checkpoint_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    item = json.loads(line)
                    if item.get("uid"):
                        completed[item["uid"]] = item
                except (json.JSONDecodeError, KeyError, TypeError):
                    continue
    pending = [row for row in data if row["uid"] not in completed]
    if max_new is not None:
        pending = pending[:max_new]
    print(f"{split_name}: 저장된 완료 {len(completed)}/{len(data)} | 이번 실행 {len(pending)}")

    result_buffer, error_buffer = [], []
    run_errors = []
    started = time.time()

    def flush_results():
        if result_buffer:
            with open(checkpoint_path, "a", encoding="utf-8") as out:
                out.writelines(json.dumps(x, ensure_ascii=False) + "\n" for x in result_buffer)
            result_buffer.clear()
        if error_buffer:
            with open(error_path, "a", encoding="utf-8") as errout:
                errout.writelines(json.dumps(x, ensure_ascii=False) + "\n" for x in error_buffer)
            error_buffer.clear()
        _flush_hop2_cache_buffer()

    for row in tqdm(pending, desc=f"{split_name} v5 2-hop 생성"):
        failure = None
        for attempt in range(2):
            try:
                result = build_explicit_two_hop_v5(row)
                result["uid"] = row["uid"]
                result["label"] = row["label"]
                result["supporting_facts"] = row["supporting_facts"]
                result_buffer.append(result)
                completed[row["uid"]] = result
                failure = None
                break
            except Exception as exc:
                failure = {"uid": row.get("uid"), "claim": row.get("claim"), "attempt": attempt + 1, "error": repr(exc)}
                if attempt == 0:
                    time.sleep(1)
        if failure is not None:
            run_errors.append(failure)
            error_buffer.append(failure)
        if len(result_buffer) + len(error_buffer) >= checkpoint_every:
            flush_results()
    flush_results()

    ordered = [completed[row["uid"]] for row in data if row["uid"] in completed]
    elapsed = time.time() - started
    processed = len(pending)
    rate = processed / elapsed if elapsed else 0
    print(f"{split_name}: 결과 {len(ordered)}/{len(data)} | 오류 {len(run_errors)} | {elapsed/60:.1f}분 | {rate:.2f} claim/s")
    print("체크포인트:", checkpoint_path)
    print("문장 캐시:", HOP2_CACHE_PATH)
    if run_errors:
        print("오류 예시:", run_errors[:2])
    return ordered, run_errors

# 이미 완료된 UID는 건너뛰고 체크포인트에서 이어서 생성합니다.
MAX_NEW_PER_SPLIT = None
train_two_hop_v5, train_errors_v5 = build_dataset_resumable(train_data, "train", MAX_NEW_PER_SPLIT)
val_two_hop_v5, val_errors_v5 = build_dataset_resumable(val_data, "validation", MAX_NEW_PER_SPLIT)


## 8. 최종 판정기 학습: v5 Top-5 + oversampling

In [ ]:
# 28번: 저장된 v5 근거로 판정기를 학습하고 v5 체크포인트를 만듭니다.
import random
import torch
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder, InputExample

HOP1_K = 5
HOP2_K = 5

def make_pair_example_v5(result):
    lines = []
    for i, item in enumerate(result.get("hop1", [])[:HOP1_K], 1):
        lines.append(f"Hop 1 Evidence {i} [{item['title']}, 문장 {item['sentence_id']}]: {item['sentence']}")
    for i, item in enumerate(result.get("hop2", [])[:HOP2_K], 1):
        lines.append(f"Hop 2 Evidence {i} [{item['title']}, 문장 {item['sentence_id']}]: {item['sentence']}")
    label_text = result["label"]
    return {
        "claim": result["claim"],
        "evidence": "\n".join(lines),
        "label": 1 if label_text == "SUPPORTED" else 0,
        "label_text": label_text,
    }

train_pair_data_v5 = [make_pair_example_v5(x) for x in train_two_hop_v5]
val_pair_data_v5 = [make_pair_example_v5(x) for x in val_two_hop_v5]
print("Train/val 입력:", len(train_pair_data_v5), len(val_pair_data_v5))
print("Train labels:", Counter(x["label_text"] for x in train_pair_data_v5))

supported = [x for x in train_pair_data_v5 if x["label"] == 1]
not_supported = [x for x in train_pair_data_v5 if x["label"] == 0]
random.seed(42)
target_size = max(len(supported), len(not_supported))
balanced_train = (random.choices(supported, k=target_size) + random.choices(not_supported, k=target_size))
random.shuffle(balanced_train)
print("Oversampling 후:", Counter(x["label_text"] for x in balanced_train))

train_examples = [InputExample(texts=[x["claim"], x["evidence"]], label=float(x["label"])) for x in balanced_train]
train_loader = DataLoader(train_examples, shuffle=True, batch_size=16)

MODEL_DIR = DRIVE_ROOT / "hover_final_verifier_v5"
verifier_v5 = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    num_labels=1,
    max_length=512,
    device=device,
)
verifier_v5.fit(
    train_dataloader=train_loader,
    epochs=1,
    warmup_steps=max(1, int(len(train_loader) * 0.1)),
    optimizer_params={"lr": 2e-5},
    output_path=str(MODEL_DIR),
    use_amp=(device == "cuda"),
    show_progress_bar=True,
)
print("Drive 체크포인트:", MODEL_DIR, MODEL_DIR.exists())

## 9. 검증 평가 및 추론 함수

In [ ]:
# 30번: v5 체크포인트를 validation으로 평가하고 추론 함수를 준비합니다.
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

val_pairs = [[x["claim"], x["evidence"]] for x in val_pair_data_v5]
y_true = np.asarray([x["label"] for x in val_pair_data_v5])
raw_scores = np.asarray(verifier_v5.predict(val_pairs, batch_size=32, show_progress_bar=True)).reshape(-1)
prob_supported = 1 / (1 + np.exp(-raw_scores))

FINAL_THRESHOLD = 0.60
y_pred = (prob_supported >= FINAL_THRESHOLD).astype(int)
print("threshold:", FINAL_THRESHOLD)
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Macro-F1:", f1_score(y_true, y_pred, average="macro"))
print(classification_report(y_true, y_pred, target_names=["NOT_SUPPORTED", "SUPPORTED"], digits=4))
print("혼동행렬:\n", confusion_matrix(y_true, y_pred))

def predict_claim(claim):
    '''현재 TF-IDF 검색 목록에 포함된 2-hop HoVer claim을 판정합니다.'''
    result = build_explicit_two_hop_v5({"claim": claim})
    evidence_lines = []
    for i, item in enumerate(result.get("hop1", [])[:HOP1_K], 1):
        evidence_lines.append({"hop": 1, "rank": i, **item})
    for i, item in enumerate(result.get("hop2", [])[:HOP2_K], 1):
        evidence_lines.append({"hop": 2, "rank": i, **item})
    evidence_text = "\n".join(
        f"Hop {x['hop']} Evidence {x['rank']} [{x['title']}, 문장 {x['sentence_id']}]: {x['sentence']}"
        for x in evidence_lines
    )
    score = float(np.asarray(verifier_v5.predict([[claim, evidence_text]], show_progress_bar=False)).reshape(-1)[0])
    probability = float(1 / (1 + np.exp(-score)))
    return {
        "claim": claim,
        "label": "SUPPORTED" if probability >= FINAL_THRESHOLD else "NOT_SUPPORTED",
        "supported_probability": probability,
        "threshold": FINAL_THRESHOLD,
        "hop1": result.get("hop1", []),
        "entities": result.get("entities", []),
        "hop2": result.get("hop2", []),
    }

sample_claim = val_data[0]["claim"]
sample_prediction = predict_claim(sample_claim)
print("추론 함수 예시:", sample_prediction["label"], sample_prediction["claim"])

## 완료 후 확인할 점

- 학습 입력 생성에서 오류가 0인지 확인하세요. 오류 샘플이 있으면 결과 수가 7,241 / 1,811보다 작아질 수 있습니다.
- 모델 체크포인트 폴더가 `/content/drive/MyDrive/hover_final_verifier_v5`에 생겼는지 확인하세요. Drive에 저장된 모델은 런타임 초기화 후에도 다시 불러올 수 있습니다.
- 이 노트북은 baseline 재현과 `predict_claim()` 추론까지 포함합니다. iOS 앱에서 호출할 HTTP API와 외부 접속 터널은 별도 배포 단계입니다. 특히 임의의 새 claim을 받으려면 HoVer에 미리 저장된 TF-IDF Top-50 목록이 아닌 실제 검색기를 추가해야 합니다.

In [ ]:
# 32번: v5 판정기 체크포인트를 Drive에 저장합니다.
from pathlib import Path

MODEL_DIR = DRIVE_ROOT / "hover_final_verifier_v5"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
verifier_v5.model.save_pretrained(str(MODEL_DIR))
verifier_v5.tokenizer.save_pretrained(str(MODEL_DIR))
assert (MODEL_DIR / "config.json").is_file(), f"체크포인트 저장 확인 실패: {MODEL_DIR}"
print("v5 체크포인트 저장 완료:", MODEL_DIR)
print("파일:", sorted(p.name for p in MODEL_DIR.iterdir()))

In [ ]:
# 33번: v5 validation에서 Hop 1과 중복되는 Hop 2 근거를 제거해 성능을 비교합니다.
# Hop 1 문장과 겹치는 Hop 2 입력을 제거했을 때의 검증 비교
import random
import unicodedata
import numpy as np
import torch
from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder, InputExample
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

H1_K, H2_K, EVAL_THRESHOLD = 5, 5, 0.60

def _norm_fact_text(text):
    return " ".join(unicodedata.normalize("NFKC", str(text)).casefold().split())

def _without_hop1_duplicates(row):
    hop1 = list(row.get("hop1", [])[:H1_K])
    hop1_fact_keys = {(normalize_title(x["title"]), int(x["sentence_id"])) for x in hop1}
    hop1_texts = {_norm_fact_text(x["sentence"]) for x in hop1}
    original_hop2 = list(row.get("hop2", [])[:H2_K])
    kept_hop2 = [
        item for item in original_hop2
        if (normalize_title(item["title"]), int(item["sentence_id"])) not in hop1_fact_keys
        and _norm_fact_text(item["sentence"]) not in hop1_texts
    ]
    return {**row, "hop1": hop1, "hop2": kept_hop2}, len(original_hop2) - len(kept_hop2)

def _prepare_rows(rows):
    prepared, removed, affected = [], 0, 0
    for row in rows:
        new_row, count = _without_hop1_duplicates(row)
        prepared.append(new_row)
        removed += count
        affected += int(count > 0)
    return prepared, removed, affected

train_dedup_rows, train_removed, train_affected = _prepare_rows(train_two_hop_v5)
val_dedup_rows, val_removed, val_affected = _prepare_rows(val_two_hop_v5)
print(f"중복 제거 | train: {train_affected}/{len(train_two_hop_v5)}건 영향, {train_removed}개 문장 제거")
print(f"중복 제거 | validation: {val_affected}/{len(val_two_hop_v5)}건 영향, {val_removed}개 문장 제거")

def _to_pairs(rows):
    pairs, labels = [], []
    for row in rows:
        lines = []
        for hop, key in ((1, "hop1"), (2, "hop2")):
            for rank, item in enumerate(row.get(key, [])[:H1_K if hop == 1 else H2_K], 1):
                lines.append(f"Hop {hop} Evidence {rank} [{item['title']}, 문장 {item['sentence_id']}]: {item['sentence']}")
        pairs.append([row["claim"], "\n".join(lines)])
        labels.append(1 if row["label"] == "SUPPORTED" else 0)
    return pairs, np.asarray(labels)

train_base_pairs, y_train = _to_pairs(train_two_hop_v5)
val_base_pairs, y_val = _to_pairs(val_two_hop_v5)
train_dedup_pairs, _ = _to_pairs(train_dedup_rows)
val_dedup_pairs, _ = _to_pairs(val_dedup_rows)

def _metrics(model, pairs, labels, name):
    raw = np.asarray(model.predict(pairs, batch_size=32, show_progress_bar=False)).reshape(-1)
    probability = 1 / (1 + np.exp(-raw))
    prediction = (probability >= EVAL_THRESHOLD).astype(int)
    result = {
        "accuracy": accuracy_score(labels, prediction),
        "macro_f1": f1_score(labels, prediction, average="macro"),
        "confusion": confusion_matrix(labels, prediction),
    }
    print(f"{name}: Accuracy={result['accuracy']:.4f} | Macro-F1={result['macro_f1']:.4f} | confusion=\n{result['confusion']}")
    return result

# 기존 체크포인트로 입력 중복만 제거했을 때의 변화
checkpoint_original = _metrics(verifier_v5, val_base_pairs, y_val, "기존 v5 · 원본 validation")
checkpoint_dedup = _metrics(verifier_v5, val_dedup_pairs, y_val, "기존 v5 · 중복 제거 validation")

def _train_one_epoch(train_pairs, labels, seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    supported = [i for i, label in enumerate(labels) if label == 1]
    not_supported = [i for i, label in enumerate(labels) if label == 0]
    target = max(len(supported), len(not_supported))
    rng = random.Random(seed)
    sampled = rng.choices(supported, k=target) + rng.choices(not_supported, k=target)
    rng.shuffle(sampled)
    examples = [InputExample(texts=train_pairs[i], label=float(labels[i])) for i in sampled]
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(examples, shuffle=True, batch_size=16, generator=generator)
    model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", num_labels=1, max_length=512, device=device)
    model.fit(
        train_dataloader=loader,
        epochs=1,
        warmup_steps=max(1, int(len(loader) * 0.1)),
        optimizer_params={"lr": 2e-5},
        output_path=None,
        use_amp=(device == "cuda"),
        show_progress_bar=True,
    )
    return model

# 동일한 사전학습 초기값·seed·학습 설정으로 baseline과 dedup 모델을 각각 비교
baseline_retrained = _train_one_epoch(train_base_pairs, y_train)
baseline_retrained_metrics = _metrics(baseline_retrained, val_base_pairs, y_val, "재학습 baseline · 원본 validation")
del baseline_retrained
torch.cuda.empty_cache() if torch.cuda.is_available() else None

dedup_retrained = _train_one_epoch(train_dedup_pairs, y_train)
dedup_retrained_metrics = _metrics(dedup_retrained, val_dedup_pairs, y_val, "재학습 dedup · 중복 제거 validation")
print("평가 완료: 기존 v5 체크포인트는 변경하거나 덮어쓰지 않았습니다.")


In [ ]:
# 34번: Hop 2 중복을 제거하고 다음 후보로 5개를 채운 뒤 validation 성능을 확인합니다.
# Hop 2 전체 문장 후보에서 Hop 1 중복을 제외한 뒤, 점수순으로 5개를 다시 채워 검증합니다.
def build_explicit_two_hop_v5_refill(row, hop1_top_k=5, hop2_top_k=5, max_hop2_titles=100):
    claim = row["claim"]
    if claim not in titles_by_claim_top50:
        raise KeyError("이 claim은 현재 TF-IDF 검색 목록에 없습니다.")

    hop1_candidates = []
    for title in titles_by_claim_top50[claim][:50]:
        for sentence_id, sentence in enumerate(get_title_sentences(title)):
            cleaned = clean_sentence(sentence)
            if cleaned:
                hop1_candidates.append({"title": title, "sentence_id": sentence_id, "sentence": cleaned})
    if not hop1_candidates:
        return {"claim": claim, "hop1": [], "entities": [], "hop2": []}

    hop1_scores = score_sentence_pairs_fast([(claim, item["sentence"]) for item in hop1_candidates])
    for item, score in zip(hop1_candidates, hop1_scores):
        item["score"] = float(score)
    hop1_ranked = sorted(hop1_candidates, key=lambda x: x["score"], reverse=True)[:hop1_top_k]

    entities = []
    for text in [claim] + [item["sentence"] for item in hop1_ranked]:
        for entity in nlp(text).ents:
            if entity.label_ in allowed_entity_labels:
                name = entity.text.strip()
                if len(name) >= 2 and name not in entities:
                    entities.append(name)

    hop2_titles, seen_titles = [], set()
    def add_title(title):
        key = normalize_title(title)
        if key and key not in seen_titles:
            seen_titles.add(key)
            hop2_titles.append(title)
    for item in hop1_ranked:
        add_title(item["title"])
    for entity_name in entities:
        for title in lookup_entity_titles(entity_name, max_results=20):
            add_title(title)
    if len(hop2_titles) < 20:
        for title in titles_by_claim_top50[claim][:50]:
            add_title(title)
    hop2_titles = hop2_titles[:max_hop2_titles]

    hop1_fact_keys = {(normalize_title(item["title"]), int(item["sentence_id"])) for item in hop1_ranked}
    hop1_sentence_texts = {_norm_fact_text(item["sentence"]) for item in hop1_ranked}
    hop2_candidates, seen_facts = [], set()
    for title in hop2_titles:
        for sentence_id, sentence in enumerate(get_title_sentences(title)):
            fact = (normalize_title(title), sentence_id)
            cleaned = clean_sentence(sentence)
            if fact in seen_facts or not cleaned:
                continue
            seen_facts.add(fact)
            if fact in hop1_fact_keys or _norm_fact_text(cleaned) in hop1_sentence_texts:
                continue
            hop2_candidates.append({"title": title, "sentence_id": sentence_id, "sentence": cleaned})

    if not hop2_candidates:
        return {"claim": claim, "hop1": hop1_ranked, "entities": entities, "hop2": []}
    hop1_text = "\n".join(item["sentence"] for item in hop1_ranked)
    hop2_query = f"Claim: {claim}\nHop 1 evidence:\n{hop1_text}"
    hop2_scores = score_sentence_pairs_fast([(hop2_query, item["sentence"]) for item in hop2_candidates])
    for item, score in zip(hop2_candidates, hop2_scores):
        item["score"] = float(score)
    hop2_ranked = sorted(hop2_candidates, key=lambda x: x["score"], reverse=True)[:hop2_top_k]
    return {"claim": claim, "hop1": hop1_ranked, "entities": entities, "hop2": hop2_ranked}

refill_val_results, refill_val_errors = [], []
for row in tqdm(val_data, desc="Hop2 중복 제거 + 다음 후보 채우기 (validation)"):
    try:
        result = build_explicit_two_hop_v5_refill(row)
        result["uid"] = row["uid"]
        result["label"] = row["label"]
        result["supporting_facts"] = row["supporting_facts"]
        refill_val_results.append(result)
    except Exception as exc:
        refill_val_errors.append({"uid": row.get("uid"), "claim": row.get("claim"), "error": repr(exc)})

print("재검색 validation 결과/오류:", len(refill_val_results), len(refill_val_errors))
if refill_val_errors:
    print("오류 예시:", refill_val_errors[:3])
    raise RuntimeError("오류가 남아 있어 성능 비교를 중단합니다.")

refill_h2_counts = [len(row.get("hop2", [])) for row in refill_val_results]
print("Hop 2 후보 수 평균:", round(float(np.mean(refill_h2_counts)), 2), "| 5개 확보:", sum(n == 5 for n in refill_h2_counts), "/", len(refill_h2_counts))
refill_val_pairs, refill_y = _to_pairs(refill_val_results)
assert np.array_equal(refill_y, y_val), "원본 validation label 순서와 다릅니다."
refill_metrics = _metrics(verifier_v5, refill_val_pairs, refill_y, "기존 v5 · Hop2 재검색 후 5개 채움")
print("원본 기준: Accuracy=0.6897 | Macro-F1=0.6327")
print("기존 체크포인트는 그대로 두었습니다.")


In [ ]:
# 35번: Hop 2 보충 검색 결과로 v6를 학습·평가하고 체크포인트를 저장합니다.
# HoVer v6 학습 셀. 실행 전 1~34번 준비/검색 셀을 먼저 실행하세요.
import random
from pathlib import Path
import numpy as np
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

required = ["train_data", "val_data", "build_explicit_two_hop_v5_refill", "_norm_fact_text", "DRIVE_ROOT", "device"]
missing = [name for name in required if name not in globals()]
if missing: raise RuntimeError("준비 셀을 먼저 실행하세요. 누락: " + ", ".join(missing))
build_explicit_two_hop_v6_refill = build_explicit_two_hop_v5_refill

def build_v6_dataset(rows, split_name):
    results, errors = [], []
    for row in tqdm(rows, desc=f"v6 {split_name}"):
        try:
            result = build_explicit_two_hop_v6_refill(row)
            result.update(uid=row.get("uid"), label=row["label"], supporting_facts=row.get("supporting_facts", []))
            results.append(result)
        except Exception as exc:
            errors.append({"uid": row.get("uid"), "claim": row.get("claim"), "error": repr(exc)})
    print(f"{split_name}: {len(results)}/{len(rows)} 결과, {len(errors)} 오류")
    if errors: print(errors[:3]); raise RuntimeError("근거 생성 오류로 학습 중단")
    return results

train_two_hop_v6 = build_v6_dataset(train_data, "train")
val_two_hop_v6 = build_v6_dataset(val_data, "validation")

def make_pair_v6(row):
    lines = []
    for hop, key in ((1, "hop1"), (2, "hop2")):
        for rank, item in enumerate(row.get(key, [])[:5], 1):
            lines.append(f"Hop {hop} Evidence {rank} [{item['title']}, 문장 {item['sentence_id']}]: {item['sentence']}")
    return [row["claim"], "\n".join(lines)], int(row["label"] == "SUPPORTED")

train_pairs_v6 = [make_pair_v6(row) for row in train_two_hop_v6]
val_pairs_v6 = [make_pair_v6(row) for row in val_two_hop_v6]
print("Train/validation:", len(train_pairs_v6), len(val_pairs_v6))

random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)
pos = [i for i, (_, y) in enumerate(train_pairs_v6) if y == 1]
neg = [i for i, (_, y) in enumerate(train_pairs_v6) if y == 0]
if not pos or not neg: raise RuntimeError("두 label 모두 필요합니다.")
target = max(len(pos), len(neg)); rng = random.Random(42)
indices = rng.choices(pos, k=target) + rng.choices(neg, k=target); rng.shuffle(indices)
examples = [InputExample(texts=train_pairs_v6[i][0], label=float(train_pairs_v6[i][1])) for i in indices]
loader = DataLoader(examples, shuffle=True, batch_size=16)
MODEL_DIR_V6 = Path(DRIVE_ROOT) / "hover_final_verifier_v6_hop2_refill"
verifier_v6 = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", num_labels=1, max_length=512, device=device)
verifier_v6.fit(train_dataloader=loader, epochs=1, warmup_steps=max(1, int(len(loader)*0.1)), optimizer_params={"lr":2e-5}, output_path=None, use_amp=(device=="cuda"), show_progress_bar=True)

y = np.asarray([label for _, label in val_pairs_v6])
raw = np.asarray(verifier_v6.predict([pair for pair, _ in val_pairs_v6], batch_size=32, show_progress_bar=True, activation_fn=torch.nn.Identity())).reshape(-1)
prob = 1/(1+np.exp(-np.clip(raw, -80, 80))); pred = (prob >= 0.60).astype(int)
print("v6 threshold=0.60 Accuracy:", accuracy_score(y,pred), "Macro-F1:", f1_score(y,pred,average="macro"))
print(classification_report(y,pred,target_names=["NOT_SUPPORTED","SUPPORTED"],digits=4))
print("Confusion matrix:\n", confusion_matrix(y,pred))
MODEL_DIR_V6.mkdir(parents=True, exist_ok=True)
verifier_v6.model.save_pretrained(str(MODEL_DIR_V6)); verifier_v6.tokenizer.save_pretrained(str(MODEL_DIR_V6))
assert (MODEL_DIR_V6/"config.json").is_file()
print("v6 checkpoint saved:", MODEL_DIR_V6)